In [69]:
import pandas as pd
import os
import io
import re
import paramiko

In [70]:
def ConnectSFTP(server, username, password, port=22):
    """
    Establece una conexión SFTP mediante SSH.

    Retorna:
        ssh: cliente SSH necesario para cerrar correctamente la conexión.
        sftp: cliente utilizado para navegar y transferir archivos.
    """
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    ssh.connect(
        hostname=server,
        port=port,
        username=username,
        password=password,
        timeout=60,
    )

    sftp = ssh.open_sftp()

    print(f"Conectado mediante SFTP a {server}:{port}")
    return ssh, sftp

def UploadCsvToSFTP(df, path):
    """
    Convierte un DataFrame a CSV en memoria y lo sube mediante SFTP.
    """
    csv_buffer = io.BytesIO()
    df.to_csv(
        csv_buffer,
        index=False,
        encoding="utf-8"
    )
    csv_buffer.seek(0)
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    # remote_path = f"riecs/pruebas2/csv/{path}"
    try:
        sftp.putfo(csv_buffer, path)
        print("Archivo subido correctamente:", path)
    finally:
        sftp.close()
        ssh.close()

def ReadCsvFromSFTP(remote_file_path):
    """
    Lee un archivo CSV desde un servidor SFTP y lo carga
    en un DataFrame de pandas.

    Parámetros:
        remote_file_path (str):
            Ruta completa del archivo CSV dentro del servidor.

    Retorna:
        pandas.DataFrame:
            DataFrame con el contenido del archivo CSV.
    """
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    try:
        with sftp.open(remote_file_path, mode="rb") as remote_file:
            df = pd.read_csv(remote_file)
        return df
    finally:
        sftp.close()
        ssh.close()

In [71]:
def ordenar_sustancias(fila, columnas_edad):
    edades_validas = fila[columnas_edad].dropna()
    edades_validas = edades_validas[(edades_validas > 0) & (edades_validas < 99)]
    columnas_ordenadas = edades_validas.sort_values().index

    sustancias_ordenadas = [col.replace("EdadInicio_", "") for col in columnas_ordenadas]
    return sustancias_ordenadas

def Columns_orden(df):
    df['Orden1'] = df['OrdenSustancia'].apply(lambda x: x[0] if len(x) > 0 else None)
    df['Orden2'] = df['OrdenSustancia'].apply(lambda x: x[1] if len(x) > 1 else None)
    df['Orden3'] = df['OrdenSustancia'].apply(lambda x: x[2] if len(x) > 2 else None)
    df['Orden4'] = df['OrdenSustancia'].apply(lambda x: x[3] if len(x) > 3 else None)
    df['Orden5'] = df['OrdenSustancia'].apply(lambda x: x[4] if len(x) > 4 else None)
    return df

def Orden1_Orden2 (df):
    dict_sustancias = {'sust':'sustancia','from': [], 'to': [], 'value': []}
    sustancia = ['Tabaco','Alcohol','Marihuana','Metanfetaminas','Otras Sustancias','Cocaína','Inhalables','Alucinógenos','Medicamentos','Opioides','Nuevas Sustancias Psicoactivas','ProblemasDeSaludMental','Ninguna']
    for sust in sustancia: 
        for sust2 in sustancia: 
            if sust != sust2: 
                count = df[(df['Orden1'] == sust) & (df['Orden2'] == sust2)].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '1')
                    dict_sustancias['to'].append(sust2 + '2')
                    dict_sustancias['value'].append(count)
            else:
                count = df[(df['Orden1'] == sust) & (df['Orden2'].isna())].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '1')
                    dict_sustancias['to'].append('Ninguna2')
                    dict_sustancias['value'].append(count)
    df_orden1_2 = pd.DataFrame(dict_sustancias)
    return df_orden1_2

def Orden2_Orden3 (df):
    dict_sustancias = {'sust':'sustancia','from': [], 'to': [], 'value': []}
    sustancia = ['Tabaco','Alcohol','Marihuana','Metanfetaminas','Otras Sustancias','Cocaína','Inhalables','Alucinógenos','Medicamentos','Opioides','Nuevas Sustancias Psicoactivas','ProblemasDeSaludMental','Ninguna']
    for sust in sustancia: 
        for sust2 in sustancia: 
            if sust != sust2: 
                count = df[(df['Orden2'] == sust) & (df['Orden3'] == sust2)].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '2')
                    dict_sustancias['to'].append(sust2 + '3')
                    dict_sustancias['value'].append(count)
            else:
                count = df[(df['Orden2'] == sust) & (df['Orden3'].isna())].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '2')
                    dict_sustancias['to'].append('Ninguna3')
                    dict_sustancias['value'].append(count)
    df_orden2_3 = pd.DataFrame(dict_sustancias)
    return df_orden2_3

def Orden3_Orden4 (df):
    dict_sustancias = {'sust':'sustancia','from': [], 'to': [], 'value': []}
    sustancia = ['Tabaco','Alcohol','Marihuana','Metanfetaminas','Otras Sustancias','Cocaína','Inhalables','Alucinógenos','Medicamentos','Opioides','Nuevas Sustancias Psicoactivas','ProblemasDeSaludMental','Ninguna']
    for sust in sustancia: 
        for sust2 in sustancia: 
            if sust != sust2: 
                count = df[(df['Orden3'] == sust) & (df['Orden4'] == sust2)].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '3')
                    dict_sustancias['to'].append(sust2 + '4')
                    dict_sustancias['value'].append(count)
            else:
                count = df[(df['Orden3'] == sust) & (df['Orden4'].isna())].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '3')
                    dict_sustancias['to'].append('Ninguna4')
                    dict_sustancias['value'].append(count)
    df_orden3_4 = pd.DataFrame(dict_sustancias)
    return df_orden3_4

def Orden4_Orden5 (df):
    dict_sustancias = {'sust':'sustancia','from': [], 'to': [], 'value': []}
    sustancia = ['Tabaco','Alcohol','Marihuana','Metanfetaminas','Otras Sustancias','Cocaína','Inhalables','Alucinógenos','Medicamentos','Opioides','Nuevas Sustancias Psicoactivas','ProblemasDeSaludMental','Ninguna']
    for sust in sustancia: 
        for sust2 in sustancia: 
            if sust != sust2: 
                count = df[(df['Orden4'] == sust) & (df['Orden5'] == sust2)].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '4')
                    dict_sustancias['to'].append(sust2 + '5')
                    dict_sustancias['value'].append(count)
            else:
                count = df[(df['Orden4'] == sust) & (df['Orden5'].isna())].shape[0] 
                if count > 0: 
                    dict_sustancias['from'].append(sust + '4')
                    dict_sustancias['to'].append('Ninguna5')
                    dict_sustancias['value'].append(count)
    df_orden4_5 = pd.DataFrame(dict_sustancias)
    return df_orden4_5

def main():
    df = ReadCsvFromSFTP("riecs/pruebas2/csv/data_Sankey/edad_inicio_GPOS-AV-SLI.csv") 
    columnas_edad = [col for col in df.columns if col.startswith("EdadInicio_")]
    df[columnas_edad] = df[columnas_edad].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)
    df = df[(df[columnas_edad].ge(1)& df[columnas_edad].lt(99)).any(axis=1)].copy()
    print(len(df))
    df["OrdenSustancia"] = df.apply(ordenar_sustancias, axis=1, columnas_edad=columnas_edad)
    df = Columns_orden(df)
    df_orden1_2 = Orden1_Orden2(df)
    df_orden2_3 = Orden2_Orden3(df)
    df_orden3_4 = Orden3_Orden4(df)
    df_orden4_5 = Orden4_Orden5(df)
    df_concat = pd.concat([df_orden1_2, df_orden2_3, df_orden3_4, df_orden4_5], ignore_index=True)
    return df_concat

In [72]:
df= main()

Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_26628\3510262834.py:70: DtypeWarning: Columns (0,21,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


540309


In [74]:
df.to_csv("sankey.csv", index=False)

In [75]:
UploadCsvToSFTP(df, f'riecs/pruebas2/csv/Sankey/sankey.csv')

Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Sankey/sankey.csv
